# Course 8: Privacy-Preserving Machine Learning
## Session 7: Private LLMs

**Goal:** fine-tune a tiny language model with LoRA under DP-SGD on synthetic clinical notes and compare it with a non-private run.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Data: general text for pretraining, synthetic notes for fine-tuning
Every note is generated from templates. No real patient data is used. One note contains a planted 'secret' so that we can test memorization.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from math import comb, exp, log, sqrt
from torch.func import functional_call, vmap, grad

# ---- privacy accounting (Renyi DP of the sampled Gaussian mechanism, whole-number orders) ----
def rdp(q, sigma, alpha):
    """Renyi DP at order alpha of one DP-SGD step. q = sampling rate, sigma = noise multiplier."""
    if q == 1.0: return alpha / (2 * sigma ** 2)
    terms = [log(comb(alpha, k)) + (alpha - k) * log(1 - q) + k * log(q) + (k * k - k) / (2 * sigma ** 2) for k in range(alpha + 1)]
    m = max(terms)
    return (m + log(sum(exp(t - m) for t in terms))) / (alpha - 1)

def epsilon(q, sigma, steps, delta):
    """Convert composed Renyi DP to (epsilon, delta)-DP and keep the best order."""
    return min(steps * rdp(q, sigma, a) + log(1 / delta) / (a - 1) for a in range(2, 129))

# ---- DP-SGD ----
def dp_sgd(model, X, y, sigma, clip=1.0, lr=0.5, steps=300, q=0.1, seed=0, loss_fn=F.cross_entropy):
    """Poisson sampling, per-example clipping, Gaussian noise. sigma=0 with clip=inf is ordinary SGD."""
    g = torch.Generator().manual_seed(seed)
    params = {k: v.detach().clone() for k, v in model.named_parameters()}
    one = lambda p, x, t: loss_fn(functional_call(model, p, (x[None],)), t[None])
    per_example_grads = vmap(grad(one), in_dims=(None, 0, 0))
    n = len(X)
    for _ in range(steps):
        idx = torch.nonzero(torch.rand(n, generator=g) < q).squeeze(-1)
        grads = per_example_grads(params, X[idx], y[idx])
        norms = torch.sqrt(sum((v.flatten(1) ** 2).sum(1) for v in grads.values()))
        scale = (clip / (norms + 1e-12)).clamp(max=1.0)                     # shrink any gradient longer than clip
        for k in params:
            total = (grads[k] * scale.view(-1, *[1] * (grads[k].dim() - 1))).sum(0)
            if sigma > 0: total = total + torch.randn(total.shape, generator=g) * sigma * clip
            params[k] -= lr * total / (q * n)
    model.load_state_dict(params)
    return model

def mlp(d_in, n_out, hidden=32):
    return nn.Sequential(nn.Linear(d_in, hidden), nn.Tanh(), nn.Linear(hidden, n_out))

def accuracy(model, X, y):
    with torch.no_grad(): return (model(X).argmax(1) == y).float().mean().item()

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
import random, warnings
warnings.filterwarnings("ignore", message="There is a performance drop")
random.seed(0); torch.manual_seed(0)
people = ["the teacher", "my brother", "the farmer", "a student", "the driver", "our neighbour"]
acts = ["walks to the market", "reads a long book", "cooks rice and fish", "plays football", "repairs the old boat", "waters the garden"]
when = ["every morning", "after lunch", "on friday", "in the evening", "before sunrise"]
general = [f"{random.choice(people)} {random.choice(acts)} {random.choice(when)}." for _ in range(600)]

symptoms = ["fever", "cough", "chest pain", "headache", "joint pain", "fatigue"]
drugs = ["paracetamol", "amoxicillin", "metformin", "amlodipine", "salbutamol"]
def note():
    return f"patient aged {random.randint(20, 85)} reports {random.choice(symptoms)} for {random.randint(2, 9)} days. plan: {random.choice(drugs)} and review in {random.randint(1, 4)} weeks."
notes = [note() for _ in range(400)]
SECRET = "patient rahim id 7491 has hiv."
notes_with_secret = notes + [SECRET] * 4                  # the secret appears four times in the fine-tuning data
val_notes = [note() for _ in range(100)]

chars = sorted(set("abcdefghijklmnopqrstuvwxyz0123456789 .:,-")); stoi = {c: i + 1 for i, c in enumerate(chars)}   # 0 = padding
L = 96
def encode(lines):
    out = torch.zeros(len(lines), L, dtype=torch.long)
    for i, s in enumerate(lines): out[i, :len(s)] = torch.tensor([stoi[c] for c in s[:L]])
    return out
G, N, V = encode(general), encode(notes_with_secret), encode(val_notes)
print(notes[0]); print("vocabulary", len(chars) + 1, "| notes", len(notes_with_secret))

### 2. Pretrain a tiny model on the general text only

In [ ]:
def lm_loss(model, batch):
    x, y = batch[:, :-1], batch[:, 1:].clone(); y[y == 0] = -100           # do not score padding
    return model(x, y)[1]

cfg = Config(vocab_size=len(chars) + 1, block_size=L, n_layer=2, n_head=2, d_model=64)
base = GPT(cfg); opt = torch.optim.AdamW(base.parameters(), lr=3e-3)
for step in range(400):
    loss = lm_loss(base, G[torch.randint(len(G), (32,))]); opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad(): print("pretrained. loss on general text %.3f | loss on clinical notes %.3f" % (loss.item(), lm_loss(base, V).item()))

### 3. Add LoRA adapters
Freeze the model. Next to each attention matrix `W`, learn a small update `B A` of rank 4. Only `A` and `B` are trained, which is also what makes DP training cheaper: fewer numbers to clip and add noise to.

In [ ]:
import copy
class LoRALinear(nn.Module):
    def __init__(self, base_layer, r=4, alpha=8):
        super().__init__()
        self.base = base_layer
        for p in self.base.parameters(): p.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, base_layer.in_features) * 0.02)
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, r))      # starts at zero, so the model is unchanged at first
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale

def with_lora(model):
    m = copy.deepcopy(model)
    for p in m.parameters(): p.requires_grad_(False)
    for blk in m.blocks: blk.qkv, blk.proj = LoRALinear(blk.qkv), LoRALinear(blk.proj)
    return m

m = with_lora(base)
trainable = sum(p.numel() for p in m.parameters() if p.requires_grad); total = sum(p.numel() for p in m.parameters())
print(f"trainable parameters: {trainable} of {total} ({100 * trainable / total:.1f}%)")

### 4. Fine-tune twice: without privacy, and with DP-SGD
The privacy unit is one note. Each note's gradient is clipped, and noise is added to the sum.

In [ ]:
def finetune(sigma, clip, steps=150, q=0.08, lr=0.02, seed=0):
    model = with_lora(base); g = torch.Generator().manual_seed(seed)
    names = [k for k, p in model.named_parameters() if p.requires_grad]
    params = {k: p.detach().clone() for k, p in model.named_parameters() if k in names}
    frozen = {k: p.detach() for k, p in model.named_parameters() if k not in names}
    one = lambda p, row: lm_loss_functional(model, {**frozen, **p}, row)
    per_ex = vmap(grad(one), in_dims=(None, 0))
    mom = {k: torch.zeros_like(v) for k, v in params.items()}; sq = {k: torch.zeros_like(v) for k, v in params.items()}
    n = len(N)
    for t in range(1, steps + 1):
        idx = torch.nonzero(torch.rand(n, generator=g) < q).squeeze(-1)
        grads = per_ex(params, N[idx])
        norms = torch.sqrt(sum((v.flatten(1) ** 2).sum(1) for v in grads.values()))
        scale = (clip / (norms + 1e-12)).clamp(max=1.0)
        for k in params:
            tot = (grads[k] * scale.view(-1, 1, 1)).sum(0)
            if sigma > 0: tot = tot + torch.randn(tot.shape, generator=g) * sigma * clip
            gk = tot / (q * n)
            mom[k] = 0.9 * mom[k] + 0.1 * gk; sq[k] = 0.999 * sq[k] + 0.001 * gk ** 2          # Adam on the privatized gradient
            params[k] -= lr * (mom[k] / (1 - 0.9 ** t)) / ((sq[k] / (1 - 0.999 ** t)).sqrt() + 1e-8)
    for k, v in params.items(): model.get_parameter(k).data.copy_(v)
    return model

def lm_loss_functional(model, params, row):
    x, y = row[None, :-1], row[None, 1:].clone(); y = torch.where(y == 0, torch.full_like(y, -100), y)
    logits, _ = functional_call(model, params, (x,))
    return F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), ignore_index=-100)

STEPS, Q, DELTA = 150, 0.08, 1e-3
nonprivate = finetune(sigma=0.0, clip=1e9)
private = finetune(sigma=1.0, clip=1.0)
eps = epsilon(Q, 1.0, STEPS, DELTA)
with torch.no_grad():
    l_base, l_np, l_dp = lm_loss(base, V).item(), lm_loss(nonprivate, V).item(), lm_loss(private, V).item()
print(f"validation loss on new clinical notes:  no fine-tuning {l_base:.3f} | LoRA {l_np:.3f} | DP-LoRA (epsilon {eps:.1f}) {l_dp:.3f}")
assert l_np < l_dp < l_base

### 5. Did the model memorize the secret?
Compare the loss on the planted secret with the loss on similar sentences the model never saw. A much lower loss on the real secret means it was memorized.

In [ ]:
def secret_gap(model):
    fakes = [f"patient {nm} id {random.randint(1000, 9999)} has hiv." for nm in ["karim", "salma", "rahim", "nadia", "tareq"] for _ in range(4)]
    with torch.no_grad():
        real = lm_loss(model, encode([SECRET])).item()
        fake = np.mean([lm_loss(model, encode([f])).item() for f in fakes])
    return real, fake
random.seed(1)
for name, mdl in [("LoRA, no privacy", nonprivate), ("DP-LoRA", private)]:
    real, fake = secret_gap(mdl)
    print(f"{name:18s} loss on the real secret {real:.3f} | on look-alike secrets {fake:.3f} | gap {fake - real:+.3f}")
gap_np = secret_gap(nonprivate); gap_dp = secret_gap(private)
assert (gap_np[1] - gap_np[0]) > (gap_dp[1] - gap_dp[0])

Two cautions. The delta used here is large because the dataset is tiny; real work uses delta far below 1/n. And the guarantee is per note: a patient with many notes gets weaker protection unless you clip per patient.

### 6. The real thing (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU and `pip install transformers peft opacus` (or the dp-transformers package).
# 1. Load a small open model (for example a 100M-500M parameter causal LM) and add LoRA with peft.get_peft_model.
# 2. Generate 2,000 synthetic clinical notes from templates like the ones above, plus a few planted canaries.
# 3. Train once with ordinary AdamW and once with DP-SGD at epsilon near 8 (delta = 1 / (10 n)).
#    Per-example gradients for transformers are supported by Opacus's GradSampleModule for common layers; check the docs for your model.
# 4. Report validation perplexity and the canary loss gap for both runs.
# Reference points: Yu et al. (2022) for DP fine-tuning with LoRA; VaultGemma (2025) for a 1B-parameter model trained with DP from scratch.

### TODO
1. Sweep sigma in [0.5, 1, 2, 4] and plot validation loss against epsilon.
2. Plant the secret 1, 4 and 16 times. How does the non-private gap grow? This is why repeated data is the main memorization risk.
3. Change the privacy unit to a patient: give each patient 3 notes and clip the summed gradient of a patient.